### Imports

In [1]:
import sys
sys.dont_write_bytecode = True


import torch
import numpy as np
import random
import os

def set_seeds(seed_value=42):
    """Sets seeds for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)

set_seeds(42) 

import json
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import warnings
import logging
from datetime import datetime
warnings.filterwarnings('ignore')

from model import get_model
from config import CFG
from dataset import get_dataset_class
from transform import get_transforms
from runner import run_baseline, run_lodo

torch.manual_seed(CFG["system"]["seed"])
np.random.seed(CFG["system"]["seed"])

device = CFG["system"]["device"]
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

DS = "VLCS"
MODEL_NAME = "resnet18"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu126 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1129 09:59:05.682000 6896 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda
PyTorch: 2.8.0+cu126


### DataLoading

In [2]:
train_transform, test_transform = get_transforms(img_size=224, augment=False, use_imagenet_norm=False)

DatasetClass = get_dataset_class(DS)

ld = DatasetClass(
    data_root=CFG["datasets"][DS]["root"],
    transform=train_transform,
    batch_size=CFG["train"]["batch_size"]
)

print("\nData loaders ready!")


Data loaders ready!


### Logging

In [3]:
dataset_name = DS
base_dir = os.path.join(os.getcwd(), dataset_name)
subdirs = ["logs", "checkpoints", "plots"]

for sub in subdirs:
    os.makedirs(os.path.join(base_dir, sub), exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = os.path.join(base_dir, "logs", f"train_{timestamp}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(f"{dataset_name}_logger")

logger.info(f"Initialized experiment directories for {dataset_name}")
logger.info(f"Logs: {os.path.join(base_dir, 'logs')}")
logger.info(f"Checkpoints: {os.path.join(base_dir, 'checkpoints')}")
logger.info(f"Plots: {os.path.join(base_dir, 'plots')}")

2025-11-29 09:59:07,035 | INFO | Initialized experiment directories for VLCS
2025-11-29 09:59:07,035 | INFO | Logs: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\VLCS\logs
2025-11-29 09:59:07,035 | INFO | Checkpoints: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\VLCS\checkpoints
2025-11-29 09:59:07,035 | INFO | Plots: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\VLCS\plots


### Setup

In [4]:
domains = CFG["datasets"][DS]["domains"]
loaders = {d: {"train": ld.get_dataloader(d, train=True), "val": ld.get_dataloader(d, train=False)} for d in domains}
ckpt_root = os.path.join(base_dir, "checkpoints")
log_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
model_factory = lambda cfg, dataset_key: get_model(cfg,dataset=DS)
optimizer_fn = lambda model: optim.AdamW(model.parameters(), lr=CFG["train"]["lr"], weight_decay=CFG["train"].get("weight_decay", 0.01))
device = CFG["system"]["device"]
epochs = CFG["train"]["epochs"]


{
  "lodo_results": {
    "art_painting": 0.8341463414634146,
    "cartoon": 0.7974413646055437,
    "photo": 0.9580838323353293,
    "sketch": 0.6017811704834606
  },
  "timestamp": "20251004_020611"
}

### Leave One Domain Out

In [5]:
lodo_results, lodo_mean, lodo_summary = run_lodo(
    model_fn=model_factory,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    ckpt_root=ckpt_root,
    log_dir=log_dir,
    epochs=epochs
)

2025-11-29 09:59:07,256 | INFO | === LODO: Leaving out domain 'VOC2007' ===



=== LODO: Leaving out domain 'VOC2007' ===


Evaluating: 100%|██████████| 27/27 [00:08<00:00,  3.17it/s]
2025-11-29 09:59:43,466 | INFO | [VOC2007] Epoch 1/10 | Train - Loss: 0.7643, Cls: 0.7514, GRQO: 0.0129, Acc: 0.7122 | Val - Loss: 0.8831, Cls: 0.8819, GRQO: 0.0012, Acc: 0.6934
2025-11-29 09:59:43,499 | INFO | [VOC2007] New best val acc: 0.6934


[VOC2007] Epoch 1/10 | Train - Loss: 0.7643, Cls: 0.7514, GRQO: 0.0129, Acc: 0.7122 | Val - Loss: 0.8831, Cls: 0.8819, GRQO: 0.0012, Acc: 0.6934
[VOC2007] New best val acc: 0.6934


Evaluating: 100%|██████████| 27/27 [00:07<00:00,  3.38it/s]
2025-11-29 10:00:19,764 | INFO | [VOC2007] Epoch 2/10 | Train - Loss: 0.3838, Cls: 0.3819, GRQO: 0.0019, Acc: 0.8667 | Val - Loss: 0.6820, Cls: 0.6815, GRQO: 0.0005, Acc: 0.7589
2025-11-29 10:00:19,815 | INFO | [VOC2007] New best val acc: 0.7589


...

[SUN09] Epoch 8/10 | Train - Loss: 0.0032, Cls: 0.0026, GRQO: 0.0005, Acc: 0.9999 | Val - Loss: 1.3864, Cls: 1.3858, GRQO: 0.0006, Acc: 0.7093


Evaluating: 100%|██████████| 26/26 [00:13<00:00,  1.90it/s]
2025-11-29 10:30:19,050 | INFO | [SUN09] Epoch 9/10 | Train - Loss: 0.0054, Cls: 0.0050, GRQO: 0.0004, Acc: 0.9991 | Val - Loss: 1.4245, Cls: 1.4240, GRQO: 0.0005, Acc: 0.7200
2025-11-29 10:30:19,104 | INFO | [SUN09] New best val acc: 0.7200


[SUN09] Epoch 9/10 | Train - Loss: 0.0054, Cls: 0.0050, GRQO: 0.0004, Acc: 0.9991 | Val - Loss: 1.4245, Cls: 1.4240, GRQO: 0.0005, Acc: 0.7200
[SUN09] New best val acc: 0.7200


Evaluating: 100%|██████████| 26/26 [00:13<00:00,  1.92it/s]
2025-11-29 10:30:58,815 | INFO | [SUN09] Epoch 10/10 | Train - Loss: 0.0436, Cls: 0.0432, GRQO: 0.0004, Acc: 0.9850 | Val - Loss: 1.2892, Cls: 1.2887, GRQO: 0.0005, Acc: 0.6938
2025-11-29 10:30:58,815 | INFO | [SUN09] Best Acc: 0.7200
2025-11-29 10:30:58,815 | INFO | ------------------------------------------------------------
2025-11-29 10:30:58,831 | INFO | LODO finished | Mean Acc: 0.7794 | Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\VLCS\logs\lodo_summary_20251129_103058.json


[SUN09] Epoch 10/10 | Train - Loss: 0.0436, Cls: 0.0432, GRQO: 0.0004, Acc: 0.9850 | Val - Loss: 1.2892, Cls: 1.2887, GRQO: 0.0005, Acc: 0.6938
[SUN09] Best Acc: 0.7200
------------------------------------------------------------
LODO finished | Mean Acc: 0.7794
Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet18_experiments\VLCS\logs\lodo_summary_20251129_103058.json


### Baseline

In [6]:
baseline_results, baseline_mean = run_baseline(
    model_name=MODEL_NAME,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    epochs=CFG["train"]["epochs"]
)

2025-11-29 10:30:58,838 | INFO | Initializing ResNet baseline: resnet18
2025-11-29 10:30:58,917 | INFO | === Baseline LODO: Leaving out domain 'VOC2007' ===


Initializing ResNet baseline: resnet18

=== Baseline LODO: Leaving out domain 'VOC2007' ===


2025-11-29 10:31:34,878 | INFO | [VOC2007] Epoch 1/10 | Train - Loss: 0.6823, Acc: 0.7472 | Val Acc: 0.7500


[VOC2007] Epoch 1/10 | Train - Loss: 0.6823, Acc: 0.7472 | Val Acc: 0.7500


2025-11-29 10:32:10,847 | INFO | [VOC2007] Epoch 2/10 | Train - Loss: 0.2881, Acc: 0.8984 | Val Acc: 0.7387


...

[SUN09] Epoch 8/10 | Train - Loss: 0.0025, Acc: 1.0000 | Val Acc: 0.7212


2025-11-29 11:01:47,098 | INFO | [SUN09] Epoch 9/10 | Train - Loss: 0.0019, Acc: 1.0000 | Val Acc: 0.7206


[SUN09] Epoch 9/10 | Train - Loss: 0.0019, Acc: 1.0000 | Val Acc: 0.7206


2025-11-29 11:02:26,848 | INFO | [SUN09] Epoch 10/10 | Train - Loss: 0.0016, Acc: 1.0000 | Val Acc: 0.7212
2025-11-29 11:02:26,848 | INFO | [SUN09] Best Val Acc: 0.7239
2025-11-29 11:02:26,848 | INFO | ------------------------------------------------------------
2025-11-29 11:02:26,848 | INFO | Baseline LODO (resnet18) finished | Mean Acc: 0.7786


[SUN09] Epoch 10/10 | Train - Loss: 0.0016, Acc: 1.0000 | Val Acc: 0.7212
[SUN09] Best Val Acc: 0.7239
------------------------------------------------------------
Baseline LODO (resnet18) finished | Mean Acc: 0.7786
